In [5]:
# =========================
# Cardialyse ECG Processing Pipeline
# Full workflow: ECG Image → Signal → R-Peaks → HRV Features
# =========================

# 1️⃣ Install required packages (run once)
# !pip install numpy pandas matplotlib opencv-python scikit-image scipy neurokit2

# =========================
# 2️⃣ Imports
# =========================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
from skimage.filters import threshold_otsu
from scipy.signal import butter, filtfilt
import neurokit2 as nk
import warnings
warnings.filterwarnings('ignore')

# =========================
# 3️⃣ Folder Paths & Output
# =========================
folders = {
    'Normal': r'D:\Cardialyse_Major_Project\Normal Person ECG Images (284x12=3408)',
    'Abnormal': r'D:\Cardialyse_Major_Project\ECG Images of Patient that have abnormal heartbeat (233x12=2796)',
    'MI': r'D:\Cardialyse_Major_Project\ECG Images of Myocardial Infarction Patients (240x12=2880)',
    'History_MI': r'D:\Cardialyse_Major_Project\ECG Images of Patient that have History of MI (172x12=2064)'
}


output_dir = r"D:\Cardialyse_Heart_Disease_Major_Project"
os.makedirs(output_dir, exist_ok=True)  # creates folder if it doesn't exist




# =========================
# 4️⃣ ECG Image → HRV Features Function
# =========================
def process_ecg_image(img_path, pixels_per_mm=10, paper_speed=25, desired_fs=500):
    """Process a single ECG image and return HRV features as a DataFrame."""
    # Load image
    img = cv2.imread(img_path)
    
    # Preprocess: grayscale + median blur
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)
    
    # Extract waveform using skeletonization
    thresh = threshold_otsu(gray)
    bw = gray < thresh  # waveform is darker
    skeleton = skeletonize(bw)
    xs, ys = [], []
    h, w = skeleton.shape
    for x in range(w):
        rows = np.where(skeleton[:, x])[0]
        if len(rows) == 0:
            continue
        y_med = int(np.median(rows))
        xs.append(x)
        ys.append(y_med)
    xs = np.array(xs)
    ys = np.array(ys)
    
    # Convert pixels → time & amplitude
    fs_est = pixels_per_mm * paper_speed
    dt = 1.0 / fs_est
    t_pixels = xs * dt
    baseline = np.median(ys)
    mV_per_pixel = 1.0 / (10.0 * pixels_per_mm)
    amps_mV = (baseline - ys) * mV_per_pixel
    t_uniform = np.arange(t_pixels.min(), t_pixels.max(), 1.0/desired_fs)
    amps_uniform = np.interp(t_uniform, t_pixels, amps_mV)
    
    # Bandpass Filter 0.5–40 Hz
    def butter_bandpass(lowcut, highcut, fs, order=3):
        nyq = 0.5 * fs
        b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
        return b, a
    b, a = butter_bandpass(0.5, 40, desired_fs)
    ecg_filtered = filtfilt(b, a, amps_uniform)
    
    # R-peak Detection with NeuroKit2
    ecg_cleaned = nk.ecg_clean(ecg_filtered, sampling_rate=desired_fs)
    signals, info = nk.ecg_peaks(ecg_cleaned, sampling_rate=desired_fs)
    
    # HRV Features
    hrv_features = nk.hrv(info, sampling_rate=desired_fs, show=False)
    
    # Add image path column
    hrv_features['image'] = img_path
    return hrv_features

# =========================
# 5️⃣ Loop Through All Folders & Images
# =========================
all_features = []

for label, folder in folders.items():
    for file in os.listdir(folder):
        if file.endswith(".jpg") or file.endswith(".png"):
            img_path = os.path.join(folder, file)
            try:
                features = process_ecg_image(img_path)
                features['label'] = label  # add folder/label
                all_features.append(features)
                print(f"Processed: {file}")
            except Exception as e:
                print(f"Error processing {file}: {e}")

# Combine all features into one DataFrame
df_all = pd.concat(all_features, ignore_index=True)



# Now save your file
output_csv = os.path.join(output_dir, "all_hrv_features.csv")
df_all.to_csv(output_csv, index=False)
print("All HRV features saved to:", output_csv)

# =========================
# 6️⃣ Optional: Preview first 5 rows
# =========================
print(df_all.head())


Processed: Normal(1).jpg
Processed: Normal(10).jpg
Processed: Normal(100).jpg
Processed: Normal(101).jpg
Processed: Normal(102).jpg
Processed: Normal(103).jpg
Processed: Normal(104).jpg
Processed: Normal(105).jpg
Processed: Normal(106).jpg
Processed: Normal(107).jpg
Processed: Normal(108).jpg
Processed: Normal(109).jpg
Processed: Normal(11).jpg
Processed: Normal(110).jpg
Processed: Normal(111).jpg
Processed: Normal(112).jpg
Processed: Normal(113).jpg
Processed: Normal(114).jpg
Processed: Normal(115).jpg
Processed: Normal(116).jpg
Processed: Normal(117).jpg
Processed: Normal(118).jpg
Processed: Normal(119).jpg
Processed: Normal(12).jpg
Processed: Normal(120).jpg
Processed: Normal(121).jpg
Processed: Normal(122).jpg
Processed: Normal(123).jpg
Processed: Normal(124).jpg
Processed: Normal(125).jpg
Processed: Normal(126).jpg
Processed: Normal(127).jpg
Processed: Normal(128).jpg
Processed: Normal(129).jpg
Processed: Normal(13).jpg
Processed: Normal(130).jpg
Processed: Normal(131).jpg
Process